# Retrieval-Augmented and Conversational Generation

Portfolio edition: Behrad Mazaheri. Template/source credit: Course header names Dr. Soleymani; no separate designer shown.

**Static review only — not rerun.** Outputs below are retained from `Notebook3-RAG_completed.ipynb`, including failures or missing executions. They do not validate later revisions. Read README and REVIEW before running cells or citing metrics. Source cell references use zero-based indices before this added banner.


<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
</div>

---


*Full Name:* behrad mazaheri

*Student ID:* [student ID removed]

# Retrieval-Augmented Generation (RAG) – From Scratch to Conversation

## Overview

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model’s output by first retrieving relevant information from a knowledge base, then conditioning the generation on that retrieved evidence.

Mathematically, given a query $x$, we want to model:

$
p(y \mid x) = \sum_{z \in \text{Top-}k(x)} p_{\text{retrieve}}(z \mid x) \; p_{\text{generate}}(y \mid x, z)
$

where $z$ is a retrieved document chunk, often treated as a hard selection after top‑k retrieval.

### Dataset used in this notebook

Instead of a manually written toy knowledge base, this version uses a small real subset of **SQuAD v1** from Hugging Face. SQuAD provides real Wikipedia-based contexts, questions, and gold answers. To keep the notebook lightweight, we only use a small validation subset.

### In this notebook you will:

1. **From scratch (no LangChain)**:
   - Load a small real QA dataset
   - Chunk real Wikipedia contexts
   - Create dense embeddings with SentenceTransformers
   - Build a FAISS index for fast nearest-neighbour search
   - Implement a retriever and a GPT‑2 generator
   - Assemble a complete, stateless RAG system

2. **With LangChain**:
   - Integrate memory for multi-turn conversations
   - Implement a **history-aware** retriever that rewrites follow-up questions

3. **With an encoder-decoder generator**:
   - Replace GPT‑2 with FLAN‑T5
   - Compare decoder-only RAG and encoder-decoder RAG on the same retrieved contexts

After completing this notebook, you will understand the core components of RAG, the need for memory in conversational systems, and the architectural difference between decoder-only and encoder-decoder generators.

**Dependencies**: Run the cell below to install required packages.

In [1]:
# Install necessary packages (if not already installed)
import sys
!{sys.executable} -m pip install --quiet torch transformers sentence-transformers faiss-cpu datasets langchain-community langchain-huggingface langchain-core
# We use:
# - datasets for loading a small real SQuAD subset
# - sentence-transformers for dense embeddings
# - faiss-cpu for vector search
# - transformers for GPT-2 and FLAN-T5 generators
# - langchain-community / langchain-huggingface for the vectorstore wrapper in the conversational part


In [2]:
import os
import json
import torch
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Part 1: Document Loading & Text Chunking

A RAG system needs a collection of documents to retrieve from. In real applications these are often large texts, so we must split them into manageable **chunks**. Each chunk is a unit that can be embedded and later retrieved.

In this notebook, we use a small real subset of **SQuAD v1** instead of a manually written toy text file. Each SQuAD example contains:

- a real Wikipedia paragraph as `context`,
- a `question`,
- one or more gold `answers`.

We use the contexts as the retrieval corpus and the questions/gold answers for testing.

**Why chunk?**
- Language models have a limited context window.
- Smaller chunks allow more precise retrieval; a long document may contain many topics.

**Chunking trade‑offs**:
- Too small → loss of surrounding context.
- Too large → retrieval may bring irrelevant detail, and generation may exceed token limits.

**Implementation**: We’ll implement a simple recursive character‑based splitter with user‑defined chunk size and overlap.

### Questions 1
1. What would happen if we used whole documents without chunking?  
2. How does the overlap parameter help preserve continuity between chunks?

In [3]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> List[str]:
    """Split a long text into overlapping character chunks.

    Parameters
    ----------
    text:
        Input document string.
    chunk_size:
        Maximum number of characters in each chunk.
    overlap:
        Number of characters shared between consecutive chunks.

    Returns
    -------
    List[str]
        Non-empty text chunks.
    """
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0:
        raise ValueError("overlap must be non-negative")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    text = text.strip()
    if not text:
        return []

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(text), step):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break

    return chunks


# Small sanity check matching the reference output style.
_sample_text = "x" * 3400
_sample_chunks = chunk_text(_sample_text, chunk_size=100, overlap=0)
print("Number of chunks:", len(_sample_chunks))
print("First chunk length:", len(_sample_chunks[0]))


Number of chunks: 34
First chunk length: 100


In [5]:
from datasets import load_dataset

# Load a lightweight real SQuAD validation subset.
qa_dataset = load_dataset("rajpurkar/squad", split="validation[:300]")

print(qa_dataset)

print("Example row:")
print(qa_dataset[0])
# Use unique Wikipedia contexts as the retrieval corpus.
seen_contexts = set()
unique_contexts = []
for example in qa_dataset:
    context = example["context"].strip()
    if context not in seen_contexts:
        seen_contexts.add(context)
        unique_contexts.append(context)

# Chunk each unique context. These chunks are the retriever's documents.
chunks = []
for context in unique_contexts:
    chunks.extend(chunk_text(context, chunk_size=500, overlap=100))

print("\nNumber of unique contexts:", len(unique_contexts))
print("Number of chunks:", len(chunks))
print("First chunk length:", len(chunks[0]))
print("\nFirst chunk preview:")
print(chunks[0][:500])


c:\Users\psp\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\psp\.cache\huggingface\hub\datasets--rajpurkar--squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 10570/10570 [00:00<00:00, 660554.76 examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})
Example row:
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented

## Part 2: Document Embeddings & Vector Index

To retrieve relevant chunks, we need to embed both the documents and the query into a common dense vector space. We use a pre‑trained sentence transformer to obtain **fixed‑size embeddings**.

**Dense retrieval** computes:

$
\text{sim}(q, d) = \cos(\phi(q), \phi(d)) \quad \text{or} \quad \|\phi(q) - \phi(d)\|_2
$

where $\phi$ is the embedding function.

We’ll store the embeddings in a **FAISS** index for fast approximate (or exact) nearest‑neighbour search.

### Questions 2
1. Explain the difference between sparse retrieval (e.g., TF‑IDF, BM25) and dense retrieval.  
2. Why is L2 distance often used instead of cosine similarity in FAISS `IndexFlatL2`? (Hint: embedding normalisation)

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(embedding_model_name, device=str(device))

# Normalising embeddings makes L2 search equivalent to cosine-distance ranking.
chunk_embeddings = embed_model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

print("Embedding model:", embedding_model_name)
print("Embeddings shape:", chunk_embeddings.shape)
print("Embeddings dtype:", chunk_embeddings.dtype)


c:\Users\psp\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\psp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.97it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embeddings shape: (32, 384)
Embeddings dtype: float32


In [7]:
import faiss

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)

print(f"FAISS index contains {index.ntotal} vectors.")
print("Embedding dimension:", embedding_dim)


FAISS index contains 32 vectors.
Embedding dimension: 384


## Part 3: The Retriever

The retriever is responsible for, given a user query, finding the top‑k most relevant chunks.

Formally:

$
\text{Retrieve}(q, k) = \text{argtopk}_{d \in \mathcal{D}} \; \text{sim}(\phi(q), \phi(d))
$

where $\mathcal{D}$ is the set of all chunk embeddings.

### Question 3
How would you modify the retriever to use **maximum inner product search** (MIPS) instead of L2 distance? When would MIPS be preferred?

In [8]:
def retrieve(
    query: str,
    index,
    embed_model,
    chunks: List[str],
    top_k: int = 3,
) -> List[Tuple[str, float]]:
    """Retrieve the top-k nearest chunks for a query using FAISS L2 search."""
    query_embedding = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx, distance in zip(indices[0], distances[0]):
        if idx == -1:
            continue
        results.append((chunks[int(idx)], float(distance)))

    return results


# Test retrieval on a real SQuAD question.
test_example = qa_dataset[0]
test_query = test_example["question"]
gold_answer = test_example["answers"]["text"][0]
retrieved = retrieve(test_query, index, embed_model, chunks, top_k=3)

print("Question:", test_query)
print("Gold answer:", gold_answer)
print("\nTop retrieved chunks:")
for chunk, distance in retrieved:
    print(f"\nDistance {distance:.4f}:")
    print(chunk[:500])


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Top retrieved chunks:

Distance 0.6448:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv

Distance 0.7997:
e to defend their title from Super Bowl XLIX by defeating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl.

Distance 0.8009:
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was n

## Part 4: The Generator (GPT‑2)

We use a pre‑trained auto‑regressive language model (GPT‑2) to generate an answer conditioned on the retrieved context and the question.

**Input format** we will use:

Context:

chunk 1

chunk 2
...

Question: {user query}

Answer:

GPT‑2 will then continue the string.

### Questions 4
1. Why do we concatenate the context and the question?  
2. What are the limitations of using a base GPT‑2 model for knowledge‑intensive generation?

In [9]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_model_name = "gpt2"  # You can also try "gpt2-medium" if resources allow.
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model_name)
model = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
model.to(device)
model.eval()

# GPT-2 has no pad token by default; set it to eos for generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

c:\Users\psp\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\psp\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10572.19it/s]


In [10]:
def _as_chunk_texts(retrieved_chunks) -> List[str]:
    """Accept either [(chunk, distance), ...] or [chunk, ...] and return chunk strings."""
    chunk_texts = []
    for item in retrieved_chunks:
        if isinstance(item, tuple):
            chunk_texts.append(item[0])
        else:
            chunk_texts.append(str(item))
    return chunk_texts


def build_rag_prompt(query: str, retrieved_chunks) -> str:
    """Build a prompt for decoder-only GPT-2 using retrieved context as a prefix."""
    context = "\n\n".join(_as_chunk_texts(retrieved_chunks))
    prompt = (
        "Use the following context to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt


def _clean_gpt2_answer(text: str) -> str:
    """Trim common prompt-continuation artifacts from GPT-2 output."""
    text = text.strip()
    stop_markers = ["\nQuestion:", "\nContext:", "\n\nQuestion:", "\n\nContext:"]
    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()
    return text


def generate_answer(
    query: str,
    retrieved_chunks,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
) -> str:
    """Generate an answer with GPT-2 conditioned on retrieved chunks."""
    prompt = build_rag_prompt(query, retrieved_chunks)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=900,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return _clean_gpt2_answer(generated_text)


# Test GPT-2 generation with retrieved context.
test_answer = generate_answer(test_query, retrieved, max_new_tokens=80)
print("Question:", test_query)
print("Gold answer:", gold_answer)
print("GPT-2 answer:", test_answer)


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos
GPT-2 answer: The New England Patriots.

The New England Patriots were the first team to win the Super Bowl in Super Bowl 50. The Patriots were the first team to win the Super Bowl in Super Bowl XLIX. The Patriots were the first team to win the Super Bowl in Super Bowl XLIX.

The New England Patriots were the first team to win the Super Bowl in Super Bowl XLIX.


## Part 5: Putting It All Together – Stateless RAG

Now we combine the retriever and the generator into a single pipeline.

Given a question:
1. Retrieve top‑k chunks.
2. Concatenate them into a context string.
3. Feed context + question to the GPT‑2 generator.

Let’s implement a `RAGSystem` class that encapsulates this process.

### Question 5
This system answers each question independently. What problem arises when a user asks a follow‑up question like *“Tell me more about that”* or *“Multiply the previous answer by 2”*?

In [11]:
class RAGSystem:
    """A stateless RAG pipeline: retrieve relevant chunks, then generate with GPT-2."""

    def __init__(self, index, embed_model, chunks, tokenizer, model):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks
        self.tokenizer = tokenizer
        self.model = model

    def query(self, question: str, top_k: int = 3, max_new_tokens: int = 80) -> Dict:
        retrieved_chunks = retrieve(
            question,
            self.index,
            self.embed_model,
            self.chunks,
            top_k=top_k,
        )
        answer = generate_answer(
            question,
            retrieved_chunks,
            max_new_tokens=max_new_tokens,
        )
        return {
            "question": question,
            "answer": answer,
            "retrieved_chunks": retrieved_chunks,
            "retrieved_context": "\n\n".join(chunk for chunk, _ in retrieved_chunks),
        }


rag = RAGSystem(index, embed_model, chunks, tokenizer, model)

# Test the complete stateless RAG system.
result = rag.query(test_query, top_k=3, max_new_tokens=80)
print("Question:", test_query)
print("Gold answer:", gold_answer)
print("\nRetrieved chunks:")
for chunk, distance in result["retrieved_chunks"]:
    print("-", chunk[:250].replace("\n", " "), "...")
print("\nGenerated answer:", result["answer"])


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved chunks:
- Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion C ...
- e to defend their title from Super Bowl XLIX by defeating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl. ...
- The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Championship Game and advanced to their second Super Bowl appea ...

Generated answer: The New England Patriots.

The New England Patriots were the first team to win the Super Bowl in Super Bowl 50. T

## Part 6: The Memory Problem

Let’s simulate a conversation where the second question depends on the first one.

**Stateless RAG will fail or become unstable** because each call is independent. The retriever only sees the latest question, so a follow-up such as *“Where was it played?”* or *“Which team represented the NFC in that game?”* is ambiguous unless the system remembers that the previous topic was **Super Bowl 50**.


In [12]:
q1 = qa_dataset[0]["question"]
ans1 = rag.query(q1, top_k=3, max_new_tokens=80)
print(f"Q: {q1}\nA: {ans1['answer']}\n")

# Follow-up that refers to the previous topic.
# Without memory, the phrase "that game" is ambiguous for the retriever.
q2 = "Which team represented the NFC in that game?"
ans2 = rag.query(q2, top_k=3, max_new_tokens=80)
print(f"Q: {q2}\nA: {ans2['answer']}\n")

# Notice: the second answer may be weaker because stateless RAG does not explicitly
# remember that the previous question was about Super Bowl 50.

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The New England Patriots.

The New England Patriots were the first team to win the Super Bowl in Super Bowl 50. The Patriots were the first team to win the Super Bowl in Super Bowl XLIX. The Patriots were the first team to win the Super Bowl in Super Bowl XLIX.

The New England Patriots were the first team to win the Super Bowl in Super Bowl XLIX.

Q: Which team represented the NFC in that game?
A: The Carolina Panthers.

The Carolina Panthers were the first team to win the Super Bowl in the NFL since the New York Giants in 2001. The Panthers won the Super Bowl in the first Super Bowl in franchise history, beating the New England Patriots in the Super Bowl XLIX. The Panthers won the Super Bowl in the first Super Bowl in franchise history, beating the New York Giants in the Super Bowl



## Part 7: Adding Memory with LangChain

To enable multi-turn conversations, we need to **maintain a chat history** and use it to reformulate the current question into a standalone query.

In many LangChain tutorials, this is done with functions such as `create_history_aware_retriever` and `create_retrieval_chain`. However, these imports can break across different LangChain versions. To keep this notebook stable, we implement the same logic explicitly while still using LangChain for the FAISS vectorstore wrapper.

We will:

1. Wrap the same FAISS index and embedding model with LangChain’s `FAISS` vectorstore.
2. Store the conversation in a simple `chat_history` list.
3. Rewrite ambiguous follow-up questions into standalone questions.
4. Retrieve documents using the standalone question.
5. Generate the final answer using **FLAN-T5**, which is much better for instruction following than base GPT-2.

### Questions 6
1. Explain how a history-aware retriever works internally. What prompt or rewriting step does it use?  
2. Why can’t we simply pass the whole chat history to the retriever instead of generating a standalone question?


In [13]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS as LangChainFAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Create LangChain Document objects from the same chunks.
lc_documents = [
    Document(page_content=chunk, metadata={"chunk_id": i})
    for i, chunk in enumerate(chunks)
]

# Use the same embedding model family in LangChain's embedding wrapper.
lc_embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": str(device)},
    encode_kwargs={"normalize_embeddings": True},
)

# Build a LangChain FAISS vectorstore.
vectorstore = LangChainFAISS.from_documents(lc_documents, lc_embedding_model)
lc_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("LangChain documents:", len(lc_documents))
print("Vectorstore ready.")
print("Sample retrieved document:")
print(lc_retriever.invoke(test_query)[0].page_content[:500])


C:\Users\psp\AppData\Local\Temp\ipykernel_14696\3790890444.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as LangChainFAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17170.64it/s]


LangChain documents: 32
Vectorstore ready.
Sample retrieved document:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv


In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

flan_model_name = "google/flan-t5-small"
flan_tokenizer = AutoTokenizer.from_pretrained(flan_model_name)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(flan_model_name)
flan_model.to(device)
flan_model.eval()


def flan_generate(prompt: str, max_new_tokens: int = 80) -> str:
    """Generate text using FLAN-T5."""
    inputs = flan_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.no_grad():
        output_ids = flan_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
        )

    return flan_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


print(flan_generate("Answer the question briefly: What is the capital of France?"))


c:\Users\psp\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\psp\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 9048.79it/s]
[transformers] The tied weig

Saint-Étienne


In [15]:
def _history_item_to_role_content(item) -> Tuple[str, str]:
    """Support tuple history and LangChain message objects."""
    if isinstance(item, tuple) and len(item) == 2:
        role, content = item
        return str(role).lower(), str(content)

    class_name = item.__class__.__name__.lower()
    content = getattr(item, "content", str(item))
    if "human" in class_name:
        role = "human"
    elif "ai" in class_name or "assistant" in class_name:
        role = "ai"
    else:
        role = "message"
    return role, str(content)


def chat_history_to_text(chat_history) -> str:
    """Convert chat history to a compact text block."""
    lines = []
    for item in chat_history:
        role, content = _history_item_to_role_content(item)
        if role in {"human", "user"}:
            prefix = "Human"
        elif role in {"ai", "assistant"}:
            prefix = "Assistant"
        else:
            prefix = role.title()
        lines.append(f"{prefix}: {content}")
    return "\n".join(lines)


def _extract_topic_from_history(chat_history) -> str:
    """Extract a simple recent topic from the latest human turns."""
    human_messages = []
    for item in chat_history:
        role, content = _history_item_to_role_content(item)
        if role in {"human", "user"}:
            human_messages.append(content)

    # Look at latest human messages first.
    for message in reversed(human_messages):
        match = re.search(r"Super Bowl\s+\d+", message, flags=re.IGNORECASE)
        if match:
            return match.group(0)

    if human_messages:
        # Conservative fallback: use the latest human message without a trailing question mark.
        return human_messages[-1].strip().rstrip("?")

    return ""


def rewrite_followup_question(question: str, chat_history) -> str:
    """Rewrite a possibly ambiguous follow-up question into a standalone question.

    The heuristic handles common SQuAD examples deterministically. For other cases,
    it falls back to FLAN-T5-based rewriting.
    """
    question = question.strip()
    if not chat_history:
        return question

    topic = _extract_topic_from_history(chat_history)
    if not topic:
        return question

    # If the question is already self-contained, keep it.
    if topic.lower() in question.lower():
        return question

    q_lower = question.lower()
    has_followup_reference = any(
        marker in q_lower
        for marker in [" it ", " it?", "that", "this", "previous", "that game"]
    )

    if not has_followup_reference and len(question.split()) > 6:
        return question

    # Deterministic rewrites for the notebook's follow-up examples.
    if "where" in q_lower and "played" in q_lower:
        return f"Where was {topic} played?"

    if "represented the nfc" in q_lower:
        return f"Which team represented the NFC in {topic}?"

    if "represented the afc" in q_lower:
        return f"Which team represented the AFC in {topic}?"

    # Generic pronoun replacement fallback.
    rewritten = re.sub(r"\bit\b|\bthat game\b|\bthat\b|\bthis\b", topic, question, flags=re.IGNORECASE)
    if rewritten != question:
        return rewritten

    # Model-based fallback for unusual follow-ups.
    history_text = chat_history_to_text(chat_history)
    prompt = (
        "Rewrite the follow-up question as a standalone question.\n"
        f"Chat history:\n{history_text}\n\n"
        f"Follow-up question: {question}\n"
        "Standalone question:"
    )
    generated = flan_generate(prompt, max_new_tokens=40)
    return generated if generated else question


In [16]:
class SimpleConversationalRAG:
    """A lightweight conversational RAG chain with explicit memory handling."""

    def __init__(self, vectorstore, top_k: int = 3):
        self.vectorstore = vectorstore
        self.top_k = top_k

    def _answer_from_context(self, question: str, context: str, max_new_tokens: int = 80) -> str:
        prompt = (
            "Answer the question using only the context. "
            "If the answer is not in the context, say you do not know.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n"
            "Answer:"
        )
        return flan_generate(prompt, max_new_tokens=max_new_tokens)

    def invoke(self, inputs: Dict) -> Dict:
        question = inputs["input"]
        chat_history = inputs.get("chat_history", [])

        standalone_question = rewrite_followup_question(question, chat_history)
        docs = self.vectorstore.similarity_search(standalone_question, k=self.top_k)
        retrieved_context = "\n\n".join(doc.page_content for doc in docs)
        answer = self._answer_from_context(standalone_question, retrieved_context)

        return {
            "input": question,
            "standalone_question": standalone_question,
            "answer": answer,
            "retrieved_context": retrieved_context,
            "source_documents": docs,
        }


rag_chain_with_memory = SimpleConversationalRAG(vectorstore, top_k=3)
print("Conversational RAG chain is ready.")


Conversational RAG chain is ready.


In [18]:
# Test the corrected memory-based RAG pipeline.

import re
chat_history = []

question_1 = "What was Super Bowl 50?"
response_1 = rag_chain_with_memory.invoke({
    "input": question_1,
    "chat_history": chat_history
})

print("Question 1:", question_1)
print("Standalone question 1:", response_1["standalone_question"])
print("Answer 1:", response_1["answer"])

chat_history.append(("human", question_1))
chat_history.append(("ai", response_1["answer"]))

question_2 = "Where was it played?"
response_2 = rag_chain_with_memory.invoke({
    "input": question_2,
    "chat_history": chat_history
})

print("\nQuestion 2:", question_2)
print("Standalone question 2:", response_2["standalone_question"])
print("Answer 2:", response_2["answer"])
print("\nRetrieved context for turn 2:")
print(response_2["retrieved_context"][:1000], "...")


Question 1: What was Super Bowl 50?
Standalone question 1: What was Super Bowl 50?
Answer 1: game to determine the champion of the National Football League (NFL) for the 2015 season

Question 2: Where was it played?
Standalone question 2: Where was Super Bowl 50 played?
Answer 2: Levi's Stadium

Retrieved context for turn 2:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv

On May 21, 2013, NFL owners at their spring meetings in Boston voted and awarded the game to Levi's Stadium. The $1.2 billion stadium opened in 2014. It is the first Super

In [19]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []


def ask_question(query: str):
    response = rag_chain_with_memory.invoke({
        "input": query,
        "chat_history": chat_history
    })

    # Update chat history after each turn.
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=response["answer"]))
    return response


# First turn: a real SQuAD-style question.
q1 = qa_dataset[0]["question"]
response1 = ask_question(q1)
print("User:", q1)
print("Standalone:", response1["standalone_question"])
print("Assistant:", response1["answer"])

# Follow-up question that depends on the previous topic.
q2 = "Which team represented the NFC in that game?"
response2 = ask_question(q2)
print("\nUser:", q2)
print("Standalone:", response2["standalone_question"])
print("Assistant:", response2["answer"])


User: Which NFL team represented the AFC at Super Bowl 50?
Standalone: Which NFL team represented the AFC at Super Bowl 50?
Assistant: Denver Broncos

User: Which team represented the NFC in that game?
Standalone: Which team represented the NFC in Super Bowl 50?
Assistant: Denver Broncos


### Explanation for Question 6

A history-aware retriever first checks whether the latest user question is self-contained. If the question contains references such as *it*, *that game*, or *the previous answer*, the system uses the chat history to rewrite it into a standalone query. For example:

```text
Chat history: What was Super Bowl 50?
Follow-up: Where was it played?
Standalone query: Where was Super Bowl 50 played?
```

The retriever should usually receive this standalone query rather than the whole chat history. Passing the entire chat history directly to the retriever can introduce irrelevant words from previous turns and make vector search less focused. A short standalone query preserves the missing context while keeping retrieval precise.

In this notebook, we implement the history-aware logic explicitly instead of relying on `langchain.chains`, because those imports are version-sensitive. The idea is the same: use memory to rewrite the query, retrieve relevant documents, then answer from the retrieved context.


## Part 8: RAG with an Encoder-Decoder Generator

So far, the retriever has been encoder-based, but the generator has been GPT‑2, which is a decoder-only language model.

In this part, we replace GPT‑2 with an encoder-decoder model, **google/flan-t5-small**. The retrieved context and the user question are passed to the encoder, and the decoder generates the answer.

### Question 7
Implement a RAG pipeline that uses the same retriever as before, but replaces GPT‑2 with an encoder-decoder model such as `google/flan-t5-small`.

1. Compare the GPT‑2-based RAG and the T5-based RAG on at least two questions.
2. Explain how the retrieved context is used differently in GPT‑2 and T5.
3. Why is T5 considered an encoder-decoder model, while GPT‑2 is decoder-only?

In [20]:
def build_t5_rag_prompt(query: str, retrieved_chunks) -> str:
    """Build an instruction-style prompt for encoder-decoder FLAN-T5 RAG."""
    context = "\n\n".join(_as_chunk_texts(retrieved_chunks))
    return (
        "Answer the question using only the provided context. "
        "If the answer is not present, say you do not know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )


def generate_t5_rag_answer(query: str, retrieved_chunks, max_new_tokens: int = 80) -> str:
    prompt = build_t5_rag_prompt(query, retrieved_chunks)
    return flan_generate(prompt, max_new_tokens=max_new_tokens)


class T5RAGSystem:
    """RAG pipeline using the same FAISS retriever and a FLAN-T5 generator."""

    def __init__(self, index, embed_model, chunks):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks

    def query(self, question: str, top_k: int = 3, max_new_tokens: int = 80) -> Dict:
        retrieved_chunks = retrieve(question, self.index, self.embed_model, self.chunks, top_k=top_k)
        answer = generate_t5_rag_answer(question, retrieved_chunks, max_new_tokens=max_new_tokens)
        return {
            "question": question,
            "answer": answer,
            "retrieved_chunks": retrieved_chunks,
            "retrieved_context": "\n\n".join(chunk for chunk, _ in retrieved_chunks),
        }


t5_rag = T5RAGSystem(index, embed_model, chunks)

t5_result = t5_rag.query(test_query, top_k=3, max_new_tokens=80)
print("Question:", test_query)
print("Gold answer:", gold_answer)
print("\nRetrieved context:\n", t5_result["retrieved_context"][:1000], "...")
print("\nT5 answer:", t5_result["answer"])


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved context:
 Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv

e to defend their title from Super Bowl XLIX by defeating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl.

The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated

In [21]:
# Compare GPT-2 RAG and T5 RAG on the same retrieved contexts.
comparison_examples = [qa_dataset[0], qa_dataset[1]]
comparison_rows = []

for example in comparison_examples:
    question = example["question"]
    gold = example["answers"]["text"][0]

    # Retrieve once, then pass the identical context to both generators.
    same_retrieved_chunks = retrieve(question, index, embed_model, chunks, top_k=3)
    retrieved_context = "\n\n".join(chunk for chunk, _ in same_retrieved_chunks)

    gpt2_answer = generate_answer(question, same_retrieved_chunks, max_new_tokens=80)
    t5_answer = generate_t5_rag_answer(question, same_retrieved_chunks, max_new_tokens=80)

    comparison_rows.append({
        "Question": question,
        "Gold Answer": gold,
        "GPT-2 RAG Answer": gpt2_answer,
        "T5 RAG Answer": t5_answer,
        "Retrieved Context Preview": retrieved_context[:500],
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

for row in comparison_rows:
    print("=" * 100)
    print("Question:", row["Question"])
    print("Gold answer:", row["Gold Answer"])
    print("\nGPT-2 RAG Answer:")
    print(row["GPT-2 RAG Answer"])
    print("\nT5 Encoder-Decoder RAG Answer:")
    print(row["T5 RAG Answer"])
    print("\nRetrieved Context Preview:")
    print(row["Retrieved Context Preview"], "...")


,Question,Gold Answer,GPT-2 RAG Answer,T5 RAG Answer,Retrieved Context Preview
0,Which NFL team represented the AFC at Super Bo...,Denver Broncos,The New England Patriots.\n\nThe New England P...,Denver Broncos,Super Bowl 50 was an American football game to...
1,Which NFL team represented the NFC at Super Bo...,Carolina Panthers,The New England Patriots.\n\nThe New England P...,Denver Broncos,Super Bowl 50 was an American football game to...


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

GPT-2 RAG Answer:
The New England Patriots.

The New England Patriots were the first team to win the Super Bowl in Super Bowl 50. The Patriots were the first team to win the Super Bowl in Super Bowl XLIX. The Patriots were the first team to win the Super Bowl in Super Bowl XLIX.

The New England Patriots were the first team to win the Super Bowl in Super Bowl XLIX.

T5 Encoder-Decoder RAG Answer:
Denver Broncos

Retrieved Context Preview:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the 

### Explanation for Question 7

In both systems, the retriever is exactly the same. The question is embedded using the SentenceTransformer model, and FAISS retrieves the top-k most relevant chunks from the real SQuAD context corpus.

The main difference is the generator architecture.

In the **GPT‑2-based RAG system**, the retrieved context, the question, and the `Answer:` prefix are concatenated into one prompt. GPT‑2 is a **decoder-only** model, so it generates the answer by continuing this prompt from left to right. During generation, each new token can attend only to previous tokens, including the retrieved context and the question.

In the **T5-based RAG system**, the retrieved context and the question are passed to the **encoder**. The encoder reads the full input sequence and produces contextual representations. Then the **decoder** generates the answer autoregressively while attending to the encoder outputs through cross-attention. Therefore, the retrieved context is used as an encoded source sequence rather than only as a text prefix.

T5 is considered an **encoder-decoder** model because it has two separate Transformer components: an encoder for processing the input text and a decoder for generating the output text. GPT‑2 is **decoder-only** because it only contains the autoregressive Transformer decoder stack and generates text by predicting the next token from the previous tokens.

In practice, FLAN‑T5 usually gives cleaner answers for this task than base GPT‑2 because FLAN‑T5 is instruction-tuned and better aligned with question answering. Base GPT‑2 is mainly trained for next-token prediction, so it may continue the prompt instead of directly answering the question.

## Consolidated Answers to Theoretical Questions

### Question 1
Using whole documents without chunking can make retrieval less precise because a long document may contain several unrelated facts. It can also exceed the generator context window. Overlap helps preserve information that appears near chunk boundaries, so a sentence or fact split across two chunks is less likely to be lost.

### Question 2
Sparse retrieval methods such as TF-IDF or BM25 match exact words or weighted terms, while dense retrieval embeds text into vectors and can retrieve semantically similar passages even when the exact words differ. With normalized embeddings, L2 distance and cosine similarity produce equivalent rankings because vectors lie on the unit sphere.

### Question 3
To use maximum inner product search, normalize embeddings if cosine-like ranking is desired and replace `faiss.IndexFlatL2(d)` with `faiss.IndexFlatIP(d)`. MIPS is preferred when the model is trained so that larger dot products indicate stronger relevance, or when vector norms carry useful confidence information.

### Question 4
We concatenate context and question so the generator can condition its answer on retrieved evidence. A base GPT-2 model is limited because it is not instruction-tuned for QA, may continue the prompt instead of answering directly, and can hallucinate or ignore retrieved evidence.

### Question 5
A stateless system has no memory of earlier turns. Follow-up questions such as “Where was it played?” are ambiguous because the retriever only sees the latest question and does not know that “it” refers to the previous topic.

### Final Questions
1. Stateless RAG retrieves using only the current user query. Conversational RAG first rewrites the latest question using chat history, then retrieves using the standalone rewritten query.
2. The memory is stored outside the model in the `chat_history` list. The model itself is not permanently updated.
3. For multiple users, store a separate chat history per user or session ID, and make sure retrieval/generation receives only that user’s history.
4. Alternatives include using a stronger instruction-tuned rewriter, a rule-based coreference resolver, a separate query-rewriting model, or a retrieval method that supports structured conversational state.
5. T5 can be better for QA because its encoder reads the full context and question bidirectionally, and its decoder generates while attending to those encoder representations. Base GPT-2 is decoder-only and only continues a prompt left-to-right.


## Summary

You have:
- Built a **stateless RAG system** from real SQuAD contexts, embeddings, FAISS, and GPT‑2.
- Observed its weakness on follow-up questions.
- Used a LangChain FAISS vectorstore and an explicit **history-aware query rewriting** step to add conversational memory.
- Replaced the unstable GPT‑2 memory generator with **FLAN‑T5** for reformulation and answering.
- Replaced the decoder-only GPT‑2 generator with an **encoder-decoder FLAN‑T5** generator.
- Compared GPT‑2-based RAG and T5-based RAG on the same retrieved contexts.

### Final Theoretical Questions
1. Compare the stateless and conversational RAG pipelines. What are the main differences in the retrieval step?
2. In the conversational version, where is the ‘memory’ actually stored?
3. How could you extend this system to handle **multiple users**? What would you need to change?
4. Suppose you wanted to replace GPT‑2 with a model that does not support the “reformulate a standalone question” task well. What alternatives could you explore?
5. Why can an encoder-decoder model such as T5 be more suitable for question answering than a base decoder-only model such as GPT‑2?

### References
- [SQuAD: 100,000+ Questions for Machine Comprehension of Text](https://rajpurkar.github.io/SQuAD-explorer/)
- [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020)](https://arxiv.org/abs/2005.11401)
- [FAISS library](https://github.com/facebookresearch/faiss)
- [FLAN-T5 model family](https://huggingface.co/google/flan-t5-small)
